# WN18RR with Global Kernel

**Goal:** Test if global kernel fixes GP-KGE on sparse-relation KGs

**Problem:** WN18RR has only 11 relations, per-relation eigendecomp fails (5/11)

**Solution:** Add global kernel that uses ALL edges regardless of relation type

**Expected:** GP-KGE AUROC should improve significantly with global kernel enabled

In [ ]:
# Setup
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !rm -rf /content/kg-bayesian-prior
    !git clone https://github.com/ChorokLeeDev/kg-bayesian-prior.git /content/kg-bayesian-prior
    !pip install -q torch-geometric gpytorch networkx pandas tqdm scikit-learn
    os.chdir('/content/kg-bayesian-prior')
    sys.path.insert(0, '/content/kg-bayesian-prior')
else:
    sys.path.insert(0, os.path.dirname(os.getcwd()))

In [ ]:
import gc, json, warnings, time
import torch
import torch.nn.functional as F
import numpy as np
from scipy import sparse
from tqdm.notebook import tqdm

from src.data.loaders import load_wn18rr
from src.models import DistMult, GPKGE
from src.utils.training import set_seed
from src.evaluation.calibration import expected_calibration_error, brier_score
from src.evaluation.ood_detection import compute_auroc, create_ood_dataset

warnings.filterwarnings('ignore')
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load WN18RR
print("Loading WN18RR...")
train_data, _, test_data = load_wn18rr()
print(f"Entities: {train_data.num_entities:,}")
print(f"Relations: {train_data.num_relations}")
print(f"Train: {len(train_data):,}, Test: {len(test_data):,}")

In [ ]:
# Config
CONFIG = {
    'embedding_dim': 100,
    'epochs': 30,
    'batch_size': 1024,
    'lr': 0.001,
    'mrr_sample': 2000,
    'ece_sample': 2000,
    'ood_sample': 2000,
    'num_eigenvectors': 50,
    'min_edges': 10,
}
print("Config:", CONFIG)

In [ ]:
def train_model(model, name, train_data, kl_weight=0.0):
    """Train a model with BCE loss"""
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])

    for ep in (pbar := tqdm(range(CONFIG['epochs']), desc=name)):
        model.train()
        loss_sum, n = 0, 0
        indices = np.random.permutation(len(train_data))

        for st in range(0, len(indices), CONFIG['batch_size']):
            batch_idx = indices[st:st+CONFIG['batch_size']]
            pos = torch.tensor(train_data.triples[batch_idx], device=device)
            neg = pos.clone()
            neg[:,2] = torch.randint(0, train_data.num_entities, (len(pos),), device=device)

            opt.zero_grad()

            if hasattr(model, 'score_triple'):
                pos_s = model.score_triple(pos[:,0], pos[:,1], pos[:,2])
                neg_s = model.score_triple(neg[:,0], neg[:,1], neg[:,2])
            else:
                pos_s = model(pos[:,0], pos[:,1], pos[:,2])
                neg_s = model(neg[:,0], neg[:,1], neg[:,2])

            bce_loss = F.binary_cross_entropy_with_logits(
                torch.cat([pos_s, neg_s]),
                torch.cat([torch.ones_like(pos_s), torch.zeros_like(neg_s)]))
            
            # Add KL if GP-KGE
            if kl_weight > 0 and hasattr(model, 'kl_divergence'):
                kl = model.kl_divergence()
                loss = bce_loss + kl_weight * kl
            else:
                loss = bce_loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            loss_sum += loss.item()
            n += 1

        pbar.set_postfix(loss=f"{loss_sum/n:.4f}")

    return model

In [ ]:
def evaluate_model(model, name, train_data, test_data):
    """Evaluate MRR, ECE, AUROC"""
    model.eval()
    results = {}

    # MRR (sampled)
    sample_idx = np.random.choice(len(test_data), min(CONFIG['mrr_sample'], len(test_data)), replace=False)
    sample = test_data.triples[sample_idx]

    ranks = []
    with torch.no_grad():
        for i in tqdm(range(0, len(sample), 100), desc=f"{name} MRR", leave=False):
            batch = sample[i:i+100]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]

            if hasattr(model, 'score_tails'):
                scores = model.score_tails(h, r)
            else:
                all_t = torch.arange(train_data.num_entities, device=device)
                scores = []
                for j in range(len(h)):
                    if hasattr(model, 'score_triple'):
                        s = model.score_triple(h[j].expand(len(all_t)), r[j].expand(len(all_t)), all_t)
                    else:
                        s = model(h[j].expand(len(all_t)), r[j].expand(len(all_t)), all_t)
                    scores.append(s)
                scores = torch.stack(scores)

            target = scores[torch.arange(len(t), device=device), t]
            ranks.extend(((scores > target.unsqueeze(1)).sum(1) + 1).cpu().tolist())

    ranks = torch.tensor(ranks, dtype=torch.float)
    results['mrr'] = (1/ranks).mean().item()
    results['hits@1'] = (ranks <= 1).float().mean().item()
    results['hits@10'] = (ranks <= 10).float().mean().item()

    # ECE (sampled)
    ece_idx = np.random.choice(len(test_data), min(CONFIG['ece_sample'], len(test_data)), replace=False)
    pos = test_data.triples[ece_idx]
    neg = np.array([[h, r, np.random.randint(train_data.num_entities)] for h,r,t in pos])
    all_t = np.vstack([pos, neg])
    labels = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])

    confs = []
    with torch.no_grad():
        for i in range(0, len(all_t), 1024):
            batch = all_t[i:i+1024]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
            if hasattr(model, 'score_triple'):
                scores = model.score_triple(h, r, t)
            else:
                scores = model(h, r, t)
            confs.append(torch.sigmoid(scores).cpu().numpy())
    conf = np.concatenate(confs)
    results['ece'], _ = expected_calibration_error(conf, labels)
    results['brier'] = brier_score(conf, labels)

    # AUROC
    id_idx = np.random.choice(len(test_data), min(CONFIG['ood_sample'], len(test_data)), replace=False)
    id_triples = test_data.triples[id_idx]
    ood_triples = create_ood_dataset(train_data, test_data, "random", CONFIG['ood_sample'])

    def get_uncertainty(triples):
        uncs = []
        with torch.no_grad():
            for i in range(0, len(triples), 1024):
                batch = triples[i:i+1024]
                h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]

                if hasattr(model, 'predict_with_uncertainty'):
                    pred = model.predict_with_uncertainty(h, r, t)
                    if isinstance(pred, dict):
                        unc = pred.get('total', pred.get('epistemic', torch.zeros(len(h))))
                    else:
                        unc = pred[1]
                    uncs.append(unc.cpu().numpy())
                else:
                    if hasattr(model, 'score_triple'):
                        s = model.score_triple(h, r, t)
                    else:
                        s = model(h, r, t)
                    p = torch.sigmoid(s)
                    unc = -p * torch.log(p + 1e-10) - (1-p) * torch.log(1-p + 1e-10)
                    uncs.append(unc.cpu().numpy())
        return np.concatenate(uncs)

    results['auroc'] = compute_auroc(get_uncertainty(id_triples), get_uncertainty(ood_triples))

    return results

## Experiment: Compare GP-KGE with/without Global Kernel

In [ ]:
set_seed(42)
all_results = {}

print("="*70)
print("WN18RR: GLOBAL KERNEL ABLATION")
print("="*70)

In [ ]:
# 1. DistMult Baseline
print("\n" + "="*50)
print("Model: DistMult (Baseline)")
print("="*50)

gc.collect()
start = time.time()

model = DistMult(train_data.num_entities, train_data.num_relations, CONFIG['embedding_dim'])
model = train_model(model, "DistMult", train_data)
results = evaluate_model(model, "DistMult", train_data, test_data)
results['time'] = time.time() - start

print(f"MRR={results['mrr']:.4f}, H@10={results['hits@10']:.4f}, ECE={results['ece']:.4f}, AUROC={results['auroc']:.4f}")
all_results['DistMult'] = results
del model

In [ ]:
# 2. GP-KGE WITHOUT Global Kernel (original, broken on WN18RR)
print("\n" + "="*50)
print("Model: GP-KGE (No Global Kernel)")
print("="*50)

gc.collect()
start = time.time()

# Import and modify kernel to disable global kernel
from src.kernels.relation_aware import RelationAwareKernel

model = GPKGE(
    train_data.num_entities,
    train_data.num_relations,
    embedding_dim=CONFIG['embedding_dim'],
    kernel_type="relation_aware",
    num_inducing=min(300, train_data.num_entities),
)

# Replace kernel with use_global_kernel=False
model.kernel = RelationAwareKernel(
    num_relations=train_data.num_relations,
    kernel_type="diffusion",
    use_global_kernel=False,  # DISABLED
)

# Set graph
model.kernel.set_graph(
    train_data.relation_adjacencies,
    train_data.num_entities,
    num_eigenvectors=CONFIG['num_eigenvectors'],
    min_edges=CONFIG['min_edges'],
)

model = train_model(model, "GP-KGE (no global)", train_data)
results = evaluate_model(model, "GP-KGE (no global)", train_data, test_data)
results['time'] = time.time() - start

print(f"MRR={results['mrr']:.4f}, H@10={results['hits@10']:.4f}, ECE={results['ece']:.4f}, AUROC={results['auroc']:.4f}")
all_results['GP-KGE (no global)'] = results
del model

In [ ]:
# 3. GP-KGE WITH Global Kernel (NEW!)
print("\n" + "="*50)
print("Model: GP-KGE (With Global Kernel)")
print("="*50)

gc.collect()
start = time.time()

model = GPKGE(
    train_data.num_entities,
    train_data.num_relations,
    embedding_dim=CONFIG['embedding_dim'],
    kernel_type="relation_aware",
    num_inducing=min(300, train_data.num_entities),
)

# Replace kernel with use_global_kernel=True (ENABLED!)
model.kernel = RelationAwareKernel(
    num_relations=train_data.num_relations,
    kernel_type="diffusion",
    use_global_kernel=True,  # ENABLED!
)

# Set graph (will build global + per-relation kernels)
model.kernel.set_graph(
    train_data.relation_adjacencies,
    train_data.num_entities,
    num_eigenvectors=CONFIG['num_eigenvectors'],
    min_edges=CONFIG['min_edges'],
)

# Initialize embeddings from graph
model._init_embeddings_from_graph()

model = train_model(model, "GP-KGE (global)", train_data)
results = evaluate_model(model, "GP-KGE (global)", train_data, test_data)
results['time'] = time.time() - start

print(f"MRR={results['mrr']:.4f}, H@10={results['hits@10']:.4f}, ECE={results['ece']:.4f}, AUROC={results['auroc']:.4f}")
all_results['GP-KGE (global)'] = results
del model

## Results Summary

In [ ]:
print("\n" + "="*80)
print("WN18RR GLOBAL KERNEL ABLATION RESULTS")
print("="*80)

print(f"\n{'Model':<25} {'MRR':>8} {'H@10':>8} {'ECE↓':>8} {'AUROC↑':>8} {'Time':>8}")
print("-"*75)
for name, r in all_results.items():
    print(f"{name:<25} {r['mrr']:>8.4f} {r['hits@10']:>8.4f} {r['ece']:>8.4f} {r['auroc']:>8.4f} {r['time']:>7.1f}s")

print("\n" + "="*80)
print("ANALYSIS")
print("="*80)

baseline = all_results['DistMult']['auroc']
no_global = all_results['GP-KGE (no global)']['auroc']
with_global = all_results['GP-KGE (global)']['auroc']

print(f"\nDistMult AUROC:           {baseline:.4f}")
print(f"GP-KGE (no global) AUROC: {no_global:.4f} ({(no_global-baseline)/baseline*100:+.1f}% vs DistMult)")
print(f"GP-KGE (global) AUROC:    {with_global:.4f} ({(with_global-baseline)/baseline*100:+.1f}% vs DistMult)")

improvement = (with_global - no_global) / no_global * 100
print(f"\nGlobal Kernel Effect: {improvement:+.1f}%")

if with_global > baseline:
    print("\n✅ SUCCESS! Global kernel fixes GP-KGE on WN18RR")
elif with_global > no_global:
    print("\n⚠️ PARTIAL: Global kernel helps, but still below DistMult")
else:
    print("\n❌ FAILED: Global kernel doesn't help")

In [ ]:
# Save results
output = {
    "dataset": "WN18RR",
    "experiment": "global_kernel_ablation",
    "config": CONFIG,
    "results": all_results,
}

os.makedirs("results", exist_ok=True)
with open("results/wn18rr_global_kernel.json", 'w') as f:
    json.dump(output, f, indent=2, default=float)
print("Saved to results/wn18rr_global_kernel.json")